# 개별종목 조합E — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합E 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합E의 피처 값만 지정합니다.
import json

COMBINATION = 'E'
FEATURE_COLUMNS = (
    'sector_ret_5',
    'sector_ret_20',
    'relative_ret_5_sector',
    'relative_ret_20_sector',
    'relative_ret_5_market',
    'sector_hv_20',
    'sector_beta_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합E 피처: ('sector_ret_5', 'sector_ret_20', 'relative_ret_5_sector', 'relative_ret_20_sector', 'relative_ret_5_market', 'sector_hv_20', 'sector_beta_60')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4831,0.5012,-0.0181,0.3489,0.3708,0.0785,0.3717,0.1625,0.2705
1,2,balanced,980,20150123,20150421,0.3695,0.3978,-0.0284,0.3234,0.3396,0.0129,0.3476,0.1706,0.2572
2,3,balanced,1210,20151228,20160328,0.3659,0.3762,-0.0103,0.3616,0.3614,0.0445,0.3617,0.3068,0.3425
3,4,balanced,1439,20161202,20170228,0.4105,0.4617,-0.0513,0.3388,0.3497,0.0309,0.3613,0.1989,0.2880
4,5,balanced,1669,20171113,20180207,0.3978,0.3901,0.0077,0.3830,0.3848,0.0787,0.3875,0.3637,0.3810
5,6,balanced,1899,20181024,20190118,0.3793,0.3725,0.0068,0.3782,0.3792,0.0713,0.3984,0.3605,0.3725
6,7,balanced,2129,20190930,20191224,0.4382,0.4781,-0.0400,0.3588,0.3700,0.0703,0.3951,0.2486,0.3299
7,8,balanced,2359,20200902,20201130,0.3895,0.3476,0.0418,0.3853,0.3869,0.0803,0.4042,0.3750,0.3832
8,9,balanced,2589,20210806,20211105,0.3588,0.3914,-0.0326,0.3535,0.3635,0.0393,0.3603,0.2902,0.3311
9,10,balanced,2818,20220714,20221012,0.3623,0.3454,0.0169,0.3620,0.3719,0.0552,0.3656,0.2823,0.3309


,OOS 폴드 평균
accuracy,0.3952
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0016
macro_f1,0.3642
balanced_accuracy,0.3715
mcc,0.0614
pr_auc_macro_ovr,0.3785
down_recall,0.2880
core_harmonic_mean,0.3366


재실행 명령: python scripts/run_stock_model_experiment.py
